In [ ]:
import sys
import csv
import os
from itertools import zip_longest
from collections import defaultdict
import time

def is_leap_year(year):
    """Determine whether a year is a leap year."""
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)

def try_file_encodings(file_path):
    """Try different encodings to read the file."""
    encodings = ['utf-8-sig', 'latin-1', 'iso-8859-1', 'windows-1252']
    for encoding in encodings:
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                return list(csv.reader(f)), encoding
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"Could not decode {file_path} with any of the tried encodings")

def initialize_counters(attacks, countries, start_year=2011, end_year=2024):
    """Initialize counters for all attack-country-date combinations."""
    c = defaultdict(lambda: defaultdict(int))
    periods = set()
    months = ['01','02','03','04','05','06','07','08','09','10','11','12']
    
    for attack in attacks:
        for country in countries:
            key = f"{attack}-{country}"
            for year in range(start_year, end_year + 1):
                for month in months:
                    if year == 2011 and int(month) < 7:
                        continue
                    max_day = 31
                    if month == '02':
                        max_day = 29 if is_leap_year(year) else 28
                    elif month in ['04', '06', '09', '11']:
                        max_day = 30
                        
                    for day in range(1, max_day + 1):
                        date = f"{day:02d}/{month}/{year}"
                        periods.add(date)
    return c, sorted(periods)

def main():
    start_time = time.time()
    
    # Configuration
    countries = ['US','GB','CA','AU','UA','RU','FR','DE','BR','CN','JP','PK',
               'KP','KR','IN','TW','NL','ES','SE','MX','IR','IL','SA','SY',
               'FI','IE','AT','NO','CH','IT','MY','EG','TR','PT','PS','AE','?','ALL']

    attacks = [
        'DDoS', 'Phishing', 'Ransomware', 'Password Attack', 'SQL Injection',
        'Account Hijacking', 'Defacement', 'Trojan', 'Vulnerability', 'Zero-day',
        'Advanced persistent threat', 'XSS', 'Malware', 'Data Breach',
        'Disinformation/Misinformation', 'Targeted Attack', 'Adware',
        'Brute Force Attack', 'Malvertising', 'Backdoor', 'Botnet',
        'Cryptojacking', 'Worms', 'Spyware', 'Unknown', 'Others'
    ]

    malwares = {'MALWARE', 'WIPER','TROJAN','DROPPER','SPYWARE','PEGASUS',
               'MALVERTIS', 'RANSOMWARE', 'VIRUS','WORM','KEYLOG', 
               'KEYSTROKE LOG', 'MALICIOUS CODE','MALICIOUS SOFTWARE',
               'ADWARE','ROOTKIT', 'BOT', 'BACKDOOR'}

    # Initialize counters through 2024
    c, periods = initialize_counters(attacks, countries)
    print(f"Initialization completed in {time.time() - start_time:.2f} seconds")

    # Define input file path
    input_path = r'Data Collection\Hackmageddon DB\20232024\Hackmageddon.csv'

    # Read and process the input data with encoding fallback
    try:
        read_start = time.time()
        data, used_encoding = try_file_encodings(input_path)
        print(f"Successfully read file with encoding: {used_encoding} in {time.time() - read_start:.2f} seconds")
    except Exception as e:
        print(f"Error reading file: {e}")
        sys.exit(1)

    # Clean data
    clean_start = time.time()
    data = [line for line in data if len(line) >= 4]
    for line in data:
        line[3] = line[3].strip()
    print(f"Data cleaning completed in {time.time() - clean_start:.2f} seconds")
    print(f"Data has {len(data)} rows after cleaning")

    # Counting logic
    count_start = time.time()
    unk = 0
    
    # Pre-compiled attack patterns
    attack_patterns = {
        'DDoS': ('DDOS', 'DENIAL OF SERVICE'),
        'Phishing': ('PHISHING',),
        'Ransomware': ('RANSOMWARE',),
        'Password Attack': ('PASSWORD',),
        'SQL Injection': ('SQLI', 'SQL I'),
        'Account Hijacking': ('ACCOUNT HIJACK', 'ACCOUNT TAKE'),
        'Defacement': ('DEFACE',),
        'Trojan': ('TROJAN', 'DROPPER'),
        'Vulnerability': ('VULNERABILITY', '0 DAY', 'ZERO DAY'),
        'Zero-day': ('0 DAY', 'ZERO DAY'),
        'Advanced persistent threat': ('APT', 'ADVANCED PERSISTENT THREAT'),
        'XSS': ('XSS', 'CROSS SITE SCRIPT'),
        'Malware': malwares,
        'Data Breach': ('BREACH', 'LEAK', 'SPILL', 'EXPOSE'),
        'Disinformation/Misinformation': ('DISINFORMATION', 'MISINFORMATION', 'FALSE INFORMATION', 'MISLEADING'),
        'Targeted Attack': ('TARGETED ATTACK',),
        'Adware': ('ADWARE',),
        'Brute Force Attack': ('BRUTE FORCE',),
        'Malvertising': ('MALVERTIS',),
        'Backdoor': ('BACKDOOR',),
        'Botnet': ('BOTNET',),
        'Cryptojacking': ('CRYPTOJACK', 'CRYPTO JACK'),
        'Worms': ('WORM',),
        'Spyware': ('SPYWARE',),
        'Unknown': ('UNKNOWN',)
    }

    for d in data:
        h_date = d[0].strip()
        if len(h_date) >= 10 and int(h_date[6:10]) == 2011 and int(h_date[3:5]) < 7:
            continue
            
        text = (d[1] + d[2]).upper().replace("-", " ")
        attack = d[2].upper().replace("-", " ")
        country_list = countries[:-1] if '>1' in d[3] else d[3].split('-')
        
        unk_increased = False
        known = False
        
        for country in country_list:
            if country not in countries:
                continue
                
            known = False
            
            # Check all attack patterns
            for attack_type, patterns in attack_patterns.items():
                if any(pattern in text for pattern in patterns):
                    c[f'{attack_type}-{country}'][h_date] += 1
                    c[f'{attack_type}-ALL'][h_date] += 1
                    known = True
                    break
            
            # Special case for malware (has its own set)
            if not known and any(malware in text for malware in malwares):
                c[f'Malware-{country}'][h_date] += 1
                c['Malware-ALL'][h_date] += 1
                known = True
            
            if not known:
                c[f'Others-{country}'][h_date] += 1 
                c['Others-ALL'][h_date] += 1 
                if not unk_increased:
                    unk += 1
                    unk_increased = True

    print(f"Counting completed in {time.time() - count_start:.2f} seconds")

    # Write output
    output_start = time.time()
    output_file = 'NoI_daily_2024.csv'
    fields = ['Attack-Country'] + periods
    
    print(f"\nWriting output to {output_file}...")
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fields)
        w.writeheader()
        for key in sorted(c.keys()):
            row = {'Attack-Country': key}
            row.update(c[key])
            w.writerow(row)

    # Transpose the data
    print("Transposing data...")
    with open(output_file, 'r', encoding='utf-8') as csvfile:
        original_data = list(csv.reader(csvfile))

    transposed_data = list(zip_longest(*original_data, fillvalue=''))

    with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(transposed_data)

    print(f"Output processing completed in {time.time() - output_start:.2f} seconds")
    print(f"Total processing time: {time.time() - start_time:.2f} seconds")
    print(f"Processing complete. Output saved to {output_file}")

if __name__ == "__main__":
    main()

1. Request permissions from the owner of the Hackmageddon site to download all data, and place the downloaded data at the root of your directory
2. Run the script 

In [ ]:
"""
The script below transforms the daily number of incidents (NoI) to monthly number of incidents
through December 2024.

Input: NoI_daily.csv (or NoI_daily_2024.csv)
Output: NoI_monthly_2024.csv
"""

import sys
import csv
import os

def is_leap_year(year):
    """Determine whether a year is a leap year."""
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)


def main():
    # File handling
    input_file = "NoI_daily_2024.csv"
    if not input_file:
        print("Error: Could not find input file (NoI_daily.csv or NoI_daily_2024.csv)")
        print("Please place the file in one of these locations:")
        print(f"- Current directory: {os.getcwd()}")
        print(f"- Data subfolder: {os.path.join(os.getcwd(), 'data')}")
        print(f"- Script directory: {os.path.dirname(__file__)}")
        print(f"- Downloads folder: {os.path.join(os.path.expanduser('~'), 'Downloads')}")
        sys.exit(1)

    output_file = 'NoI_monthly_2024.csv'
    months = ['01','02','03','04','05','06','07','08','09','10','11','12']

    # Read input data
    try:
        with open(input_file, 'r') as datafile:
            data = list(csv.reader(datafile))
    except Exception as e:
        print(f"Error reading input file: {e}")
        sys.exit(1)

    # Process data
    header = data[0]
    data = data[1:]
    col = len(header) - 1  # Automatically determine number of columns

    # Remove first column (dates)
    for row in data:
        del row[0]

    data_m = []
    index = -1
    
    # Process through 2024
    for year in range(2011, 2025):  
        for month in months:
            # Skip months before July 2011 (original filter)
            if year == 2011 and int(month) < 7:
                continue
                 
            counter = 0
            m = [f"{month}/{year}"] + [0] * col
            
            for day in range(1, 32):
                # Handle month lengths
                if month == '02' and day > 28 and not is_leap_year(year):
                    continue
                if month == '02' and day > 29:
                    continue
                if day > 30 and month in ['04', '06', '09', '11']:
                    continue
                
                date = f"{day:02d}/{month}/{year}"
                index += 1
                
                # Make sure we don't go past the end of our data
                if index >= len(data):
                    break
                
                for j in range(col):  # columns
                    try:
                        value = float(m[j+1]) + float(data[index][j])
                        m[j+1] = str(value)
                    except (IndexError, ValueError):
                        # Handle cases where data might be missing or malformed
                        continue
                        
            data_m.append(m)
    
    # Add header back
    data_m.insert(0, header)

    # Write output
    try:
        with open(output_file, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerows(data_m)
        print(f"Successfully created monthly aggregation through 2024: {output_file}")
    except Exception as e:
        print(f"Error writing output file: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()

1. Place NoI daily 2024 at the root of your cwd
2. Run the script

In [ ]:
""" Script to reorder columns in NoI_monthly_2024.csv to match the desired order. """

import pandas as pd
import numpy as np

def reorder_csv_columns(input_file, output_file):
    # Define the desired column order
    desired_order = [
        "DDoS-US", "DDoS-GB", "DDoS-CA", "DDoS-AU", "DDoS-UA", "DDoS-RU", "DDoS-FR", "DDoS-DE", "DDoS-BR", "DDoS-CN", 
        "DDoS-JP", "DDoS-PK", "DDoS-KP", "DDoS-KR", "DDoS-IN", "DDoS-TW", "DDoS-NL", "DDoS-ES", "DDoS-SE", "DDoS-MX", 
        "DDoS-IR", "DDoS-IL", "DDoS-SA", "DDoS-SY", "DDoS-FI", "DDoS-IE", "DDoS-AT", "DDoS-NO", "DDoS-CH", "DDoS-IT", 
        "DDoS-MY", "DDoS-EG", "DDoS-TR", "DDoS-PT", "DDoS-PS", "DDoS-AE", "DDoS-?", "DDoS-ALL", 
        "Phishing-US", "Phishing-GB", "Phishing-CA", "Phishing-AU", "Phishing-UA", "Phishing-RU", "Phishing-FR", "Phishing-DE", 
        "Phishing-BR", "Phishing-CN", "Phishing-JP", "Phishing-PK", "Phishing-KP", "Phishing-KR", "Phishing-IN", "Phishing-TW", 
        "Phishing-NL", "Phishing-ES", "Phishing-SE", "Phishing-MX", "Phishing-IR", "Phishing-IL", "Phishing-SA", "Phishing-SY", 
        "Phishing-FI", "Phishing-IE", "Phishing-AT", "Phishing-NO", "Phishing-CH", "Phishing-IT", "Phishing-MY", "Phishing-EG", 
        "Phishing-TR", "Phishing-PT", "Phishing-PS", "Phishing-AE", "Phishing-?", "Phishing-ALL", 
        "Ransomware-US", "Ransomware-GB", "Ransomware-CA", "Ransomware-AU", "Ransomware-UA", "Ransomware-RU", "Ransomware-FR", 
        "Ransomware-DE", "Ransomware-BR", "Ransomware-CN", "Ransomware-JP", "Ransomware-PK", "Ransomware-KP", "Ransomware-KR", 
        "Ransomware-IN", "Ransomware-TW", "Ransomware-NL", "Ransomware-ES", "Ransomware-SE", "Ransomware-MX", "Ransomware-IR", 
        "Ransomware-IL", "Ransomware-SA", "Ransomware-SY", "Ransomware-FI", "Ransomware-IE", "Ransomware-AT", "Ransomware-NO", 
        "Ransomware-CH", "Ransomware-IT", "Ransomware-MY", "Ransomware-EG", "Ransomware-TR", "Ransomware-PT", "Ransomware-PS", 
        "Ransomware-AE", "Ransomware-?", "Ransomware-ALL", 
        "Password Attack-US", "Password Attack-GB", "Password Attack-CA", "Password Attack-AU", "Password Attack-UA", 
        "Password Attack-RU", "Password Attack-FR", "Password Attack-DE", "Password Attack-BR", "Password Attack-CN", 
        "Password Attack-JP", "Password Attack-PK", "Password Attack-KP", "Password Attack-KR", "Password Attack-IN", 
        "Password Attack-TW", "Password Attack-NL", "Password Attack-ES", "Password Attack-SE", "Password Attack-MX", 
        "Password Attack-IR", "Password Attack-IL", "Password Attack-SA", "Password Attack-SY", "Password Attack-FI", 
        "Password Attack-IE", "Password Attack-AT", "Password Attack-NO", "Password Attack-CH", "Password Attack-IT", 
        "Password Attack-MY", "Password Attack-EG", "Password Attack-TR", "Password Attack-PT", "Password Attack-PS", 
        "Password Attack-AE", "Password Attack-?", "Password Attack-ALL", 
        "SQL Injection-US", "SQL Injection-GB", "SQL Injection-CA", "SQL Injection-AU", "SQL Injection-UA", "SQL Injection-RU", 
        "SQL Injection-FR", "SQL Injection-DE", "SQL Injection-BR", "SQL Injection-CN", "SQL Injection-JP", "SQL Injection-PK", 
        "SQL Injection-KP", "SQL Injection-KR", "SQL Injection-IN", "SQL Injection-TW", "SQL Injection-NL", "SQL Injection-ES", 
        "SQL Injection-SE", "SQL Injection-MX", "SQL Injection-IR", "SQL Injection-IL", "SQL Injection-SA", "SQL Injection-SY", 
        "SQL Injection-FI", "SQL Injection-IE", "SQL Injection-AT", "SQL Injection-NO", "SQL Injection-CH", "SQL Injection-IT", 
        "SQL Injection-MY", "SQL Injection-EG", "SQL Injection-TR", "SQL Injection-PT", "SQL Injection-PS", "SQL Injection-AE", 
        "SQL Injection-?", "SQL Injection-ALL", 
        "Account Hijacking-US", "Account Hijacking-GB", "Account Hijacking-CA", "Account Hijacking-AU", "Account Hijacking-UA", 
        "Account Hijacking-RU", "Account Hijacking-FR", "Account Hijacking-DE", "Account Hijacking-BR", "Account Hijacking-CN", 
        "Account Hijacking-JP", "Account Hijacking-PK", "Account Hijacking-KP", "Account Hijacking-KR", "Account Hijacking-IN", 
        "Account Hijacking-TW", "Account Hijacking-NL", "Account Hijacking-ES", "Account Hijacking-SE", "Account Hijacking-MX", 
        "Account Hijacking-IR", "Account Hijacking-IL", "Account Hijacking-SA", "Account Hijacking-SY", "Account Hijacking-FI", 
        "Account Hijacking-IE", "Account Hijacking-AT", "Account Hijacking-NO", "Account Hijacking-CH", "Account Hijacking-IT", 
        "Account Hijacking-MY", "Account Hijacking-EG", "Account Hijacking-TR", "Account Hijacking-PT", "Account Hijacking-PS", 
        "Account Hijacking-AE", "Account Hijacking-?", "Account Hijacking-ALL", 
        "Defacement-US", "Defacement-GB", "Defacement-CA", "Defacement-AU", "Defacement-UA", "Defacement-RU", "Defacement-FR", 
        "Defacement-DE", "Defacement-BR", "Defacement-CN", "Defacement-JP", "Defacement-PK", "Defacement-KP", "Defacement-KR", 
        "Defacement-IN", "Defacement-TW", "Defacement-NL", "Defacement-ES", "Defacement-SE", "Defacement-MX", "Defacement-IR", 
        "Defacement-IL", "Defacement-SA", "Defacement-SY", "Defacement-FI", "Defacement-IE", "Defacement-AT", "Defacement-NO", 
        "Defacement-CH", "Defacement-IT", "Defacement-MY", "Defacement-EG", "Defacement-TR", "Defacement-PT", "Defacement-PS", 
        "Defacement-AE", "Defacement-?", "Defacement-ALL", 
        "Trojan-US", "Trojan-GB", "Trojan-CA", "Trojan-AU", "Trojan-UA", "Trojan-RU", "Trojan-FR", "Trojan-DE", "Trojan-BR", 
        "Trojan-CN", "Trojan-JP", "Trojan-PK", "Trojan-KP", "Trojan-KR", "Trojan-IN", "Trojan-TW", "Trojan-NL", "Trojan-ES", 
        "Trojan-SE", "Trojan-MX", "Trojan-IR", "Trojan-IL", "Trojan-SA", "Trojan-SY", "Trojan-FI", "Trojan-IE", "Trojan-AT", 
        "Trojan-NO", "Trojan-CH", "Trojan-IT", "Trojan-MY", "Trojan-EG", "Trojan-TR", "Trojan-PT", "Trojan-PS", "Trojan-AE", 
        "Trojan-?", "Trojan-ALL", 
        "Vulnerability-US", "Vulnerability-GB", "Vulnerability-CA", "Vulnerability-AU", "Vulnerability-UA", "Vulnerability-RU", 
        "Vulnerability-FR", "Vulnerability-DE", "Vulnerability-BR", "Vulnerability-CN", "Vulnerability-JP", "Vulnerability-PK", 
        "Vulnerability-KP", "Vulnerability-KR", "Vulnerability-IN", "Vulnerability-TW", "Vulnerability-NL", "Vulnerability-ES", 
        "Vulnerability-SE", "Vulnerability-MX", "Vulnerability-IR", "Vulnerability-IL", "Vulnerability-SA", "Vulnerability-SY", 
        "Vulnerability-FI", "Vulnerability-IE", "Vulnerability-AT", "Vulnerability-NO", "Vulnerability-CH", "Vulnerability-IT", 
        "Vulnerability-MY", "Vulnerability-EG", "Vulnerability-TR", "Vulnerability-PT", "Vulnerability-PS", "Vulnerability-AE", 
        "Vulnerability-?", "Vulnerability-ALL", 
        "Zero-day-US", "Zero-day-GB", "Zero-day-CA", "Zero-day-AU", "Zero-day-UA", "Zero-day-RU", "Zero-day-FR", "Zero-day-DE", 
        "Zero-day-BR", "Zero-day-CN", "Zero-day-JP", "Zero-day-PK", "Zero-day-KP", "Zero-day-KR", "Zero-day-IN", "Zero-day-TW", 
        "Zero-day-NL", "Zero-day-ES", "Zero-day-SE", "Zero-day-MX", "Zero-day-IR", "Zero-day-IL", "Zero-day-SA", "Zero-day-SY", 
        "Zero-day-FI", "Zero-day-IE", "Zero-day-AT", "Zero-day-NO", "Zero-day-CH", "Zero-day-IT", "Zero-day-MY", "Zero-day-EG", 
        "Zero-day-TR", "Zero-day-PT", "Zero-day-PS", "Zero-day-AE", "Zero-day-?", "Zero-day-ALL", 
        "Advanced persistent threat-US", "Advanced persistent threat-GB", "Advanced persistent threat-CA", "Advanced persistent threat-AU", 
        "Advanced persistent threat-UA", "Advanced persistent threat-RU", "Advanced persistent threat-FR", "Advanced persistent threat-DE", 
        "Advanced persistent threat-BR", "Advanced persistent threat-CN", "Advanced persistent threat-JP", "Advanced persistent threat-PK", 
        "Advanced persistent threat-KP", "Advanced persistent threat-KR", "Advanced persistent threat-IN", "Advanced persistent threat-TW", 
        "Advanced persistent threat-NL", "Advanced persistent threat-ES", "Advanced persistent threat-SE", "Advanced persistent threat-MX", 
        "Advanced persistent threat-IR", "Advanced persistent threat-IL", "Advanced persistent threat-SA", "Advanced persistent threat-SY", 
        "Advanced persistent threat-FI", "Advanced persistent threat-IE", "Advanced persistent threat-AT", "Advanced persistent threat-NO", 
        "Advanced persistent threat-CH", "Advanced persistent threat-IT", "Advanced persistent threat-MY", "Advanced persistent threat-EG", 
        "Advanced persistent threat-TR", "Advanced persistent threat-PT", "Advanced persistent threat-PS", "Advanced persistent threat-AE", 
        "Advanced persistent threat-?", "Advanced persistent threat-ALL", 
        "XSS-US", "XSS-GB", "XSS-CA", "XSS-AU", "XSS-UA", "XSS-RU", "XSS-FR", "XSS-DE", "XSS-BR", "XSS-CN", "XSS-JP", 
        "XSS-PK", "XSS-KP", "XSS-KR", "XSS-IN", "XSS-TW", "XSS-NL", "XSS-ES", "XSS-SE", "XSS-MX", "XSS-IR", "XSS-IL", 
        "XSS-SA", "XSS-SY", "XSS-FI", "XSS-IE", "XSS-AT", "XSS-NO", "XSS-CH", "XSS-IT", "XSS-MY", "XSS-EG", "XSS-TR", 
        "XSS-PT", "XSS-PS", "XSS-AE", "XSS-?", "XSS-ALL", 
        "Malware-US", "Malware-GB", "Malware-CA", "Malware-AU", "Malware-UA", "Malware-RU", "Malware-FR", "Malware-DE", 
        "Malware-BR", "Malware-CN", "Malware-JP", "Malware-PK", "Malware-KP", "Malware-KR", "Malware-IN", "Malware-TW", 
        "Malware-NL", "Malware-ES", "Malware-SE", "Malware-MX", "Malware-IR", "Malware-IL", "Malware-SA", "Malware-SY", 
        "Malware-FI", "Malware-IE", "Malware-AT", "Malware-NO", "Malware-CH", "Malware-IT", "Malware-MY", "Malware-EG", 
        "Malware-TR", "Malware-PT", "Malware-PS", "Malware-AE", "Malware-?", "Malware-ALL", 
        "Data Breach-US", "Data Breach-GB", "Data Breach-CA", "Data Breach-AU", "Data Breach-UA", "Data Breach-RU", 
        "Data Breach-FR", "Data Breach-DE", "Data Breach-BR", "Data Breach-CN", "Data Breach-JP", "Data Breach-PK", 
        "Data Breach-KP", "Data Breach-KR", "Data Breach-IN", "Data Breach-TW", "Data Breach-NL", "Data Breach-ES", 
        "Data Breach-SE", "Data Breach-MX", "Data Breach-IR", "Data Breach-IL", "Data Breach-SA", "Data Breach-SY", 
        "Data Breach-FI", "Data Breach-IE", "Data Breach-AT", "Data Breach-NO", "Data Breach-CH", "Data Breach-IT", 
        "Data Breach-MY", "Data Breach-EG", "Data Breach-TR", "Data Breach-PT", "Data Breach-PS", "Data Breach-AE", 
        "Data Breach-?", "Data Breach-ALL", 
        "Disinformation/Misinformation-US", "Disinformation/Misinformation-GB", "Disinformation/Misinformation-CA", 
        "Disinformation/Misinformation-AU", "Disinformation/Misinformation-UA", "Disinformation/Misinformation-RU", 
        "Disinformation/Misinformation-FR", "Disinformation/Misinformation-DE", "Disinformation/Misinformation-BR", 
        "Disinformation/Misinformation-CN", "Disinformation/Misinformation-JP", "Disinformation/Misinformation-PK", 
        "Disinformation/Misinformation-KP", "Disinformation/Misinformation-KR", "Disinformation/Misinformation-IN", 
        "Disinformation/Misinformation-TW", "Disinformation/Misinformation-NL", "Disinformation/Misinformation-ES", 
        "Disinformation/Misinformation-SE", "Disinformation/Misinformation-MX", "Disinformation/Misinformation-IR", 
        "Disinformation/Misinformation-IL", "Disinformation/Misinformation-SA", "Disinformation/Misinformation-SY", 
        "Disinformation/Misinformation-FI", "Disinformation/Misinformation-IE", "Disinformation/Misinformation-AT", 
        "Disinformation/Misinformation-NO", "Disinformation/Misinformation-CH", "Disinformation/Misinformation-IT", 
        "Disinformation/Misinformation-MY", "Disinformation/Misinformation-EG", "Disinformation/Misinformation-TR", 
        "Disinformation/Misinformation-PT", "Disinformation/Misinformation-PS", "Disinformation/Misinformation-AE", 
        "Disinformation/Misinformation-?", "Disinformation/Misinformation-ALL", 
        "Targeted Attack-US", "Targeted Attack-GB", "Targeted Attack-CA", "Targeted Attack-AU", "Targeted Attack-UA", 
        "Targeted Attack-RU", "Targeted Attack-FR", "Targeted Attack-DE", "Targeted Attack-BR", "Targeted Attack-CN", 
        "Targeted Attack-JP", "Targeted Attack-PK", "Targeted Attack-KP", "Targeted Attack-KR", "Targeted Attack-IN", 
        "Targeted Attack-TW", "Targeted Attack-NL", "Targeted Attack-ES", "Targeted Attack-SE", "Targeted Attack-MX", 
        "Targeted Attack-IR", "Targeted Attack-IL", "Targeted Attack-SA", "Targeted Attack-SY", "Targeted Attack-FI", 
        "Targeted Attack-IE", "Targeted Attack-AT", "Targeted Attack-NO", "Targeted Attack-CH", "Targeted Attack-IT", 
        "Targeted Attack-MY", "Targeted Attack-EG", "Targeted Attack-TR", "Targeted Attack-PT", "Targeted Attack-PS", 
        "Targeted Attack-AE", "Targeted Attack-?", "Targeted Attack-ALL", 
        "Adware-US", "Adware-GB", "Adware-CA", "Adware-AU", "Adware-UA", "Adware-RU", "Adware-FR", "Adware-DE", 
        "Adware-BR", "Adware-CN", "Adware-JP", "Adware-PK", "Adware-KP", "Adware-KR", "Adware-IN", "Adware-TW", 
        "Adware-NL", "Adware-ES", "Adware-SE", "Adware-MX", "Adware-IR", "Adware-IL", "Adware-SA", "Adware-SY", 
        "Adware-FI", "Adware-IE", "Adware-AT", "Adware-NO", "Adware-CH", "Adware-IT", "Adware-MY", "Adware-EG", 
        "Adware-TR", "Adware-PT", "Adware-PS", "Adware-AE", "Adware-?", "Adware-ALL", 
        "Brute Force Attack-US", "Brute Force Attack-GB", "Brute Force Attack-CA", "Brute Force Attack-AU", 
        "Brute Force Attack-UA", "Brute Force Attack-RU", "Brute Force Attack-FR", "Brute Force Attack-DE", 
        "Brute Force Attack-BR", "Brute Force Attack-CN", "Brute Force Attack-JP", "Brute Force Attack-PK", 
        "Brute Force Attack-KP", "Brute Force Attack-KR", "Brute Force Attack-IN", "Brute Force Attack-TW", 
        "Brute Force Attack-NL", "Brute Force Attack-ES", "Brute Force Attack-SE", "Brute Force Attack-MX", 
        "Brute Force Attack-IR", "Brute Force Attack-IL", "Brute Force Attack-SA", "Brute Force Attack-SY", 
        "Brute Force Attack-FI", "Brute Force Attack-IE", "Brute Force Attack-AT", "Brute Force Attack-NO", 
        "Brute Force Attack-CH", "Brute Force Attack-IT", "Brute Force Attack-MY", "Brute Force Attack-EG", 
        "Brute Force Attack-TR", "Brute Force Attack-PT", "Brute Force Attack-PS", "Brute Force Attack-AE", 
        "Brute Force Attack-?", "Brute Force Attack-ALL", 
        "Malvertising-US", "Malvertising-GB", "Malvertising-CA", "Malvertising-AU", "Malvertising-UA", "Malvertising-RU", 
        "Malvertising-FR", "Malvertising-DE", "Malvertising-BR", "Malvertising-CN", "Malvertising-JP", "Malvertising-PK", 
        "Malvertising-KP", "Malvertising-KR", "Malvertising-IN", "Malvertising-TW", "Malvertising-NL", "Malvertising-ES", 
        "Malvertising-SE", "Malvertising-MX", "Malvertising-IR", "Malvertising-IL", "Malvertising-SA", "Malvertising-SY", 
        "Malvertising-FI", "Malvertising-IE", "Malvertising-AT", "Malvertising-NO", "Malvertising-CH", "Malvertising-IT", 
        "Malvertising-MY", "Malvertising-EG", "Malvertising-TR", "Malvertising-PT", "Malvertising-PS", "Malvertising-AE", 
        "Malvertising-?", "Malvertising-ALL", 
        "Backdoor-US", "Backdoor-GB", "Backdoor-CA", "Backdoor-AU", "Backdoor-UA", "Backdoor-RU", "Backdoor-FR", "Backdoor-DE", 
        "Backdoor-BR", "Backdoor-CN", "Backdoor-JP", "Backdoor-PK", "Backdoor-KP", "Backdoor-KR", "Backdoor-IN", "Backdoor-TW", 
        "Backdoor-NL", "Backdoor-ES", "Backdoor-SE", "Backdoor-MX", "Backdoor-IR", "Backdoor-IL", "Backdoor-SA", "Backdoor-SY", 
        "Backdoor-FI", "Backdoor-IE", "Backdoor-AT", "Backdoor-NO", "Backdoor-CH", "Backdoor-IT", "Backdoor-MY", "Backdoor-EG", 
        "Backdoor-TR", "Backdoor-PT", "Backdoor-PS", "Backdoor-AE", "Backdoor-?", "Backdoor-ALL", 
        "Botnet-US", "Botnet-GB", "Botnet-CA", "Botnet-AU", "Botnet-UA", "Botnet-RU", "Botnet-FR", "Botnet-DE", "Botnet-BR", 
        "Botnet-CN", "Botnet-JP", "Botnet-PK", "Botnet-KP", "Botnet-KR", "Botnet-IN", "Botnet-TW", "Botnet-NL", "Botnet-ES", 
        "Botnet-SE", "Botnet-MX", "Botnet-IR", "Botnet-IL", "Botnet-SA", "Botnet-SY", "Botnet-FI", "Botnet-IE", "Botnet-AT", 
        "Botnet-NO", "Botnet-CH", "Botnet-IT", "Botnet-MY", "Botnet-EG", "Botnet-TR", "Botnet-PT", "Botnet-PS", "Botnet-AE", 
        "Botnet-?", "Botnet-ALL", 
        "Cryptojacking-US", "Cryptojacking-GB", "Cryptojacking-CA", "Cryptojacking-AU", "Cryptojacking-UA", "Cryptojacking-RU", 
        "Cryptojacking-FR", "Cryptojacking-DE", "Cryptojacking-BR", "Cryptojacking-CN", "Cryptojacking-JP", "Cryptojacking-PK", 
        "Cryptojacking-KP", "Cryptojacking-KR", "Cryptojacking-IN", "Cryptojacking-TW", "Cryptojacking-NL", "Cryptojacking-ES", 
        "Cryptojacking-SE", "Cryptojacking-MX", "Cryptojacking-IR", "Cryptojacking-IL", "Cryptojacking-SA", "Cryptojacking-SY", 
        "Cryptojacking-FI", "Cryptojacking-IE", "Cryptojacking-AT", "Cryptojacking-NO", "Cryptojacking-CH", "Cryptojacking-IT", 
        "Cryptojacking-MY", "Cryptojacking-EG", "Cryptojacking-TR", "Cryptojacking-PT", "Cryptojacking-PS", "Cryptojacking-AE", 
        "Cryptojacking-?", "Cryptojacking-ALL", 
        "Worms-US", "Worms-GB", "Worms-CA", "Worms-AU", "Worms-UA", "Worms-RU", "Worms-FR", "Worms-DE", "Worms-BR", "Worms-CN", 
        "Worms-JP", "Worms-PK", "Worms-KP", "Worms-KR", "Worms-IN", "Worms-TW", "Worms-NL", "Worms-ES", "Worms-SE", "Worms-MX", 
        "Worms-IR", "Worms-IL", "Worms-SA", "Worms-SY", "Worms-FI", "Worms-IE", "Worms-AT", "Worms-NO", "Worms-CH", "Worms-IT", 
        "Worms-MY", "Worms-EG", "Worms-TR", "Worms-PT", "Worms-PS", "Worms-AE", "Worms-?", "Worms-ALL", 
        "Spyware-US", "Spyware-GB", "Spyware-CA", "Spyware-AU", "Spyware-UA", "Spyware-RU", "Spyware-FR", "Spyware-DE", 
        "Spyware-BR", "Spyware-CN", "Spyware-JP", "Spyware-PK", "Spyware-KP", "Spyware-KR", "Spyware-IN", "Spyware-TW", 
        "Spyware-NL", "Spyware-ES", "Spyware-SE", "Spyware-MX", "Spyware-IR", "Spyware-IL", "Spyware-SA", "Spyware-SY", 
        "Spyware-FI", "Spyware-IE", "Spyware-AT", "Spyware-NO", "Spyware-CH", "Spyware-IT", "Spyware-MY", "Spyware-EG", 
        "Spyware-TR", "Spyware-PT", "Spyware-PS", "Spyware-AE", "Spyware-?", "Spyware-ALL", 
        "Unknown-US", "Unknown-GB", "Unknown-CA", "Unknown-AU", "Unknown-UA", "Unknown-RU", "Unknown-FR", "Unknown-DE", 
        "Unknown-BR", "Unknown-CN", "Unknown-JP", "Unknown-PK", "Unknown-KP", "Unknown-KR", "Unknown-IN", "Unknown-TW", 
        "Unknown-NL", "Unknown-ES", "Unknown-SE", "Unknown-MX", "Unknown-IR", "Unknown-IL", "Unknown-SA", "Unknown-SY", 
        "Unknown-FI", "Unknown-IE", "Unknown-AT", "Unknown-NO", "Unknown-CH", "Unknown-IT", "Unknown-MY", "Unknown-EG", 
        "Unknown-TR", "Unknown-PT", "Unknown-PS", "Unknown-AE", "Unknown-?", "Unknown-ALL", 
        "Others-US", "Others-GB", "Others-CA", "Others-AU", "Others-UA", "Others-RU", "Others-FR", "Others-DE", "Others-BR", 
        "Others-CN", "Others-JP", "Others-PK", "Others-KP", "Others-KR", "Others-IN", "Others-TW", "Others-NL", "Others-ES", 
        "Others-SE", "Others-MX", "Others-IR", "Others-IL", "Others-SA", "Others-SY", "Others-FI", "Others-IE", "Others-AT", 
        "Others-NO", "Others-CH", "Others-IT", "Others-MY", "Others-EG", "Others-TR", "Others-PT", "Others-PS", "Others-AE", 
        "Others-?", "Others-ALL"
    ]
    
    # Read the CSV file
    df = pd.read_csv(input_file)
    
    # Get current columns
    current_columns = list(df.columns)
    
    print(f"Original number of columns: {len(current_columns)}")
    print(f"Desired number of columns: {len(desired_order)}")
    
    # Check for missing columns in the current CSV
    missing_columns = [col for col in desired_order if col not in current_columns]
    if missing_columns:
        print(f"Adding {len(missing_columns)} missing columns with 0 values: {missing_columns}")
        
        # Add missing columns with 0 values
        for col in missing_columns:
            df[col] = 0
    
    # Check for extra columns in the current CSV (columns not in desired order)
    extra_columns = [col for col in current_columns if col not in desired_order]
    if extra_columns:
        print(f"Warning: The following columns are not in the desired order and will be placed at the end: {extra_columns}")
    
    # Create the final column order - all desired columns, then any extra columns
    final_order = [col for col in desired_order]  # Include all desired columns
    final_order.extend(extra_columns)  # Add any extra columns at the end
    
    # Reorder the dataframe
    df_reordered = df[final_order]
    
    # Save the reordered CSV
    df_reordered.to_csv(output_file, index=False)
    
    print(f"CSV file has been reordered and saved as: {output_file}")
    print(f"Final number of columns: {len(final_order)}")
    print(f"Missing columns filled with 0 values: {len(missing_columns)}")

# Usage
if __name__ == "__main__":
    input_filename = "NoI_monthly_2024.csv"  # Replace with your input file name
    output_filename = "hackmageddon_reordered_output.csv"  # Replace with your desired output file name
    
    reorder_csv_columns(input_filename, output_filename)

1. Place NoI_monthly_2024.csv at the top of your cwd
2. Run the script 